In [1]:
import pandas as pd
import duckdb
from IPython.display import display

In [2]:
path = (r"\Users\DELL\Downloads\archive")
aisles = pd.read_csv(r"\Users\DELL\Downloads\archive\aisles.csv")
departments = pd.read_csv(r"\Users\DELL\Downloads\archive\departments.csv")
order_products = pd.read_csv(r"\Users\DELL\Downloads\archive\order_products__prior.csv")
orders = pd.read_csv(r"\Users\DELL\Downloads\archive\orders.csv")
products = pd.read_csv(r"\Users\DELL\Downloads\archive\products.csv")
order_products

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0
...,...,...,...,...
32434484,3421083,39678,6,1
32434485,3421083,11352,7,0
32434486,3421083,4600,8,0
32434487,3421083,24852,9,1


In [3]:
conn = duckdb.connect()
conn.sql(
    """
    SELECT avg(reordered) from order_products;
    """)

┌────────────────────┐
│   avg(reordered)   │
│       double       │
├────────────────────┤
│ 0.5896974667922161 │
└────────────────────┘

2. Find the top 5 most popular products based on the number of times they appear in the 
order_products table.

In [5]:
conn.sql("""
    SELECT p.product_name,COUNT(*) AS purchase_count
    FROM order_products op
    JOIN products p
    ON op.product_id = p.product_id
    GROUP BY p.product_name
    ORDER BY purchase_count DESC
    LIMIT 5;
    """)

┌────────────────────────┬────────────────┐
│      product_name      │ purchase_count │
│        varchar         │     int64      │
├────────────────────────┼────────────────┤
│ Banana                 │         472565 │
│ Bag of Organic Bananas │         379450 │
│ Organic Strawberries   │         264683 │
│ Organic Baby Spinach   │         241921 │
│ Organic Hass Avocado   │         213584 │
└────────────────────────┴────────────────┘

3. Identify the top 10 users who have placed the highest number of total orders.

In [6]:
conn.sql(
    """
    SELECT user_id,COUNT(order_id) AS total_orders
    FROM orders
    GROUP BY user_id
    ORDER BY total_orders DESC
    LIMIT 10;
    """
)

┌─────────┬──────────────┐
│ user_id │ total_orders │
│  int64  │    int64     │
├─────────┼──────────────┤
│   42971 │          100 │
│   95171 │          100 │
│   97138 │          100 │
│   94764 │          100 │
│   96577 │          100 │
│   41002 │          100 │
│   97290 │          100 │
│   41356 │          100 │
│   95062 │          100 │
│   97439 │          100 │
└─────────┴──────────────┘
  10 rows      2 columns

4. Find the top 5 products that are most frequently added to the cart first (where 
add_to_cart_order = 1).

In [8]:
conn.sql("""
    SELECT 
    p.product_name,
    COUNT(*) AS first_added_count
FROM order_products op
JOIN products p
    ON op.product_id = p.product_id
WHERE op.add_to_cart_order = 1
GROUP BY p.product_name
ORDER BY first_added_count DESC
LIMIT 5;
    """)

┌────────────────────────┬───────────────────┐
│      product_name      │ first_added_count │
│        varchar         │       int64       │
├────────────────────────┼───────────────────┤
│ Banana                 │            110916 │
│ Bag of Organic Bananas │             78988 │
│ Organic Whole Milk     │             30927 │
│ Organic Strawberries   │             27975 │
│ Organic Hass Avocado   │             24116 │
└────────────────────────┴───────────────────┘

5. Categorize orders into 'Weekend' (Days 0 and 1) vs 'Weekday' (Days 2-6) and count the total
orders for each category.

In [9]:
conn.sql("""
    SELECT 
    CASE 
        WHEN order_dow IN (0, 1) THEN 'Weekend'
        ELSE 'Weekday'
    END AS day_type,
    COUNT(*) AS total_orders
FROM orders
GROUP BY day_type;
    """)

┌──────────┬──────────────┐
│ day_type │ total_orders │
│ varchar  │    int64     │
├──────────┼──────────────┤
│ Weekday  │      2232700 │
│ Weekend  │      1188383 │
└──────────┴──────────────┘